# 01c. Multicontact extracellular stimulation and recording

In the previous tutorial, we had a single extracellular stimulating electrode positioned near a neuron. In this tutorial, we will explore how to set up more complex extracellular stimulation and recording scenarios using multiple electrodes.

```{important}
If running locally, you will need to have `ffmpeg` installed to visualize the extracellular potential recordings. You can install it using your package manager (e.g., `apt-get install ffmpeg` on Debian-based systems), via conda (`conda install -c conda-forge ffmpeg`), or download it from the [official website](https://ffmpeg.org/download.html).
```

As before. we will simulate a single myelinated axon. However, now the model will be receiving extracellular kHz stimulation from 2 point source electrodes.

In [ ]:
import dendra as dn
from dendra.models.mod import hh
from dendra.units import mm, um

model = dn.Myelinated(
    diameters=[6.0], n_node=201, node_length=1.0, celsius=6.3, v_init=-65.0
)
model.insert(hh)

Dendra makes it easy to define multiple extracellular electrodes and configure their positions, stimulation parameters, and recording settings. This allows for more realistic simulations of neural responses to complex stimulation patterns, as well as the ability to capture extracellular potentials at multiple locations around the neuron. The only diifference in code from the previous tutorial is defining multiple electrodes and setting up their parameters accordingly.

In [ ]:
s1 = dn.isotropic_point(x=-40 * mm, y=0.0 * mm, z=200.0 * um)
s2 = dn.isotropic_point(x=40 * mm, y=0.0 * mm, z=-200.0 * um)

Lets first model the effect of each individually:

In [ ]:
from dendra.units import Hz

rec = dn.callbacks.Recorder(['v'])
stim1 = dn.sin(amp=0.1, freq=100*Hz)

tstop, dt = 50.0, 0.025

rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=(s1(model), stim1),
    callbacks=[rec],
)

In [ ]:
from pathlib import Path
from movie_utils import movie_profile

out = Path("../_static/media/s1.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s1")

In [ ]:
from IPython.display import Video, display
display(Video(str(out), embed=True))

In [ ]:
stim2 = dn.sin(amp=0.1, freq=200*Hz, delay=5.0)

tstop, dt = 50.0, 0.025

rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=(s2(model), stim2),
    callbacks=[rec],
)

out = Path("../_static/media/s2.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s2")
display(Video(str(out), embed=True))


Applying both simultaneously is as simple as adding both to the `extra` argument of the `longrun` function.

In [ ]:
rec.reset()
model.initialize()
model.longrun(
    tstop=tstop,
    dt=dt,
    chunklength=100,
    extra=[(s1(model), stim1), (s2(model), stim2)],
    callbacks=[rec],
)

out = Path("../_static/media/s1_and_s2.mp4")
out.parent.mkdir(parents=True, exist_ok=True)

movie_profile(rec.numpy('v')[:, 0, :], model.x.flatten().numpy(), out, fps=30, title="s1_and_s2")
display(Video(str(out), embed=True))


Dendra will handle the superposition of the extracellular potentials from both electrodes automatically, allowing us to simulate the combined effect of multiple stimulation sources on the neuron. You can supply as many electrodes as needed in the `extra` argument, making it straightforward to set up complex extracellular stimulation scenarios.